In [ ]:
import pandas as pd
import os
import gzip
import json
import csv

os.chdir('../data/beauty')
os.getcwd()

In [ ]:
meta_path = 'meta.json.gz'

def parse(path):
  g = gzip.open(path, 'rb')
  for l in g:
    yield eval(l)

def getDF(path):
  i = 0
  df = {}
  for d in parse(path):
    df[i] = d
    i += 1
  return pd.DataFrame.from_dict(df, orient='index')

meta_df = getDF(meta_path)

valid_asins = set(meta_df['asin'].dropna().unique())
print(f"Extracted {len(valid_asins)} ASINs")

In [ ]:
input_path = '../complete.json.gz'
output_path = 'inter.csv'

with open(output_path, mode='w', newline='', encoding='utf-8') as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(['user_id', 'parent_asin', 'rating', 'timestamp'])

    with gzip.open(input_path, 'rt', encoding='utf-8') as infile:
        for line in infile:
            try:
                data = json.loads(line)
                asin = data.get('asin')
                if asin in valid_asins:
                    user_id = data.get('reviewerID')
                    rating = data.get('overall')
                    timestamp = data.get('unixReviewTime')

                    if user_id and rating and timestamp:
                        writer.writerow([user_id, asin, rating, timestamp])
            except json.JSONDecodeError:
                continue

In [ ]:
df = pd.read_csv(output_path)
df.head()